# BRGC Revision — Notebook 02: Paired significance + endpoint grouping (CPU)

**Purpose.** Convert the per-query ranks from Notebook 01 into the statistics the reviewers asked for:
- **Paired bootstrap 95% CIs for MRR differences** (B2 vs B0 = does context help; B2 vs H_ONLY = does dropping tail context hurt; B0 vs the best training-free baseline = learned vs statistics). Addresses Reviewer 2, Reviewer 3 (comment 3), Reviewer 4.
- **Context gain grouped by tail-ambiguity** (candidate relations per tail), directly testing Reviewer 3's comment 1: *does context help more when the endpoint is less informative?*

**No GPU needed.** Runs in a few minutes, on Kaggle or locally.

**On Kaggle:** attach `brgc-kg-splits` and your Notebook-01 output (the zips from Notebook 01, added as a dataset and unzipped, or uploaded as folders). This notebook auto-finds every `per_query_ranks.npz` under `/kaggle/input`, whatever the exact folder nesting Kaggle gives it (it has been tested against real Notebook-01 output for all three datasets, which nest slightly differently: CoDEx-S and FB15k-237 as `<zip_name>/<RUN>/seed_<N>/...`, CoDEx-M as `<zip_name>/CoDEx-M/strict_final/<RUN>/seed_<N>/...` — both patterns are handled).

**Locally:** set `EXTRA_LOCAL_ROOTS` below to the folder(s) holding your unzipped Notebook-01 outputs and the `brgc_kg_splits` folder — no other changes needed.

Writes `paired_significance.csv` and `endpoint_uncertainty_grouping.csv` to `OUT` (send these two files back for the revision). Already validated end-to-end on the real B0/B2/H_ONLY × 3-seed output for CoDEx-S, CoDEx-M and FB15k-237 — see the committed results in `revision_experiments/02_significance_and_grouping/results/`.

In [ ]:
# === Load per-query ranks (from Notebook 01 outputs) + KG splits ===
# ATTACH as Kaggle datasets (Add Input):
#   1) brgc-kg-splits                     (CoDEx-S/ CoDEx-M/ FB15k-237/, each with train/valid/test.txt)
#   2) your Notebook-01 output (the BRGC_perquery_<DATASET>.zip files from Notebook 01), unzipped.
# This notebook auto-finds every per_query_ranks.npz under all of INPUT_ROOTS, regardless of the
# exact nesting (Kaggle/zip tools sometimes add or drop a wrapping folder).
#
# Running locally instead of on Kaggle: add the folder(s) holding your unzipped Notebook-01 output
# and the brgc_kg_splits folder to EXTRA_LOCAL_ROOTS (absolute paths), e.g.:
#   EXTRA_LOCAL_ROOTS = [r"/path/to/revision_experiments/01_<your_nb01_output_folder>",
#                        r"/path/to/revision_experiments/kaggle_datasets/brgc_kg_splits"]
from pathlib import Path
import numpy as np, pandas as pd, json, re, collections, os

EXTRA_LOCAL_ROOTS = []  # <- fill in for a local run; leave empty on Kaggle

INPUT_ROOTS = []
kaggle_root = Path("/kaggle/input")
if kaggle_root.exists():
    INPUT_ROOTS.append(kaggle_root)
for r in EXTRA_LOCAL_ROOTS:
    rp = Path(r)
    if rp.exists():
        INPUT_ROOTS.append(rp)
if not INPUT_ROOTS:
    # last-resort fallback: search two levels up from the notebook (covers running this notebook
    # in place inside revision_experiments/02_significance_and_grouping/ with 01_... as a sibling)
    here = Path.cwd()
    for cand in [here, here.parent, here.parent.parent]:
        if cand.exists():
            INPUT_ROOTS.append(cand)
INPUT_ROOTS = list(dict.fromkeys(INPUT_ROOTS))  # dedupe, keep order
print("Searching for per_query_ranks.npz / KG splits under:", [str(r) for r in INPUT_ROOTS])

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
OUT.mkdir(parents=True, exist_ok=True)

def find_splits(name):
    for root in INPUT_ROOTS:
        for c in root.rglob(name):
            if c.is_dir() and all((c / s).exists() for s in ["train.txt", "valid.txt", "test.txt"]):
                return c
    return None

# discover per-query npz under any root; two layouts observed from real Notebook-01 output:
#   .../<DATASET>/strict_final/<RUN>/seed_<S>/per_query_ranks.npz   (e.g. CoDEx-M)
#   .../BRGC_perquery_<DATASET>/<RUN>/seed_<S>/per_query_ranks.npz  (e.g. CoDEx-S, FB15k-237)
rank_files = []
for root in INPUT_ROOTS:
    rank_files += list(root.rglob("per_query_ranks.npz"))
print("found", len(rank_files), "per_query_ranks.npz files")

rx = re.compile(r"/([^/]+)/strict_final/([^/]+)/seed_(\d+)/per_query_ranks\.npz$")
records = {}
for f in rank_files:
    s = str(f)
    m = rx.search(s)
    if not m:  # fall back: infer RUN from the immediate parent dirs, DATASET from the path string
        seed = int(re.search(r"seed_(\d+)", s).group(1))
        run  = f.parent.parent.name
        ds   = None
        for cand in ["CoDEx-S", "CoDEx-M", "FB15k-237"]:
            if cand in s:
                ds = cand
                break
    else:
        ds, run, seed = m.group(1), m.group(2), int(m.group(3))
    if ds is None:
        print("WARNING: could not infer dataset for", s, "- skipping")
        continue
    records.setdefault(ds, {}).setdefault(run, {})[seed] = f

for ds in records:
    print(ds, {r: sorted(records[ds][r]) for r in records[ds]})


In [ ]:
# === Training-free baselines: per-query filtered ranks (expected-rank, paper protocol) ===
def load_triples(p):
    T=[]
    for line in open(p, encoding="utf-8"):
        line=line.strip()
        if line:
            h,r,t=line.split("\t"); T.append((h,r,t))
    return T

def expected_rank(scores, gold, keep):
    # scores: dict r->score; keep: set of relations NOT filtered out (gold + non-known-true)
    gs=scores.get(gold, 0.0); greater=0; equal=0
    for r in keep:
        s=scores.get(r,0.0)
        if r==gold: continue
        if s>gs: greater+=1
        elif s==gs: equal+=1
    return greater + (equal/2.0) + 1.0   # expected rank under random tie-break

def baseline_perquery(ds_dir):
    tr=load_triples(ds_dir/"train.txt"); te=load_triples(ds_dir/"test.txt")
    R=sorted({r for _,r,_ in tr}); Rset=set(R); N=len(R); alpha=0.1
    glob=collections.Counter(r for _,r,_ in tr)
    hr=collections.defaultdict(collections.Counter); tc=collections.defaultdict(collections.Counter)
    for h,r,t in tr: hr[h][r]+=1; tc[t][r]+=1
    pair_true=collections.defaultdict(set)
    for h,r,t in tr: pair_true[(h,t)].add(r)
    # test gold relations restricted to training vocabulary (matches paper)
    def scores_for(h,t,kind):
        if kind=="freq": return {r: glob.get(r,0)+alpha for r in R}
        if kind=="p_r_h":
            c=hr.get(h,{}); tot=sum(c.values()); return {r:(c.get(r,0)+alpha)/(tot+alpha*N) for r in R}
        if kind=="p_r_t":
            c=tc.get(t,{}); tot=sum(c.values()); return {r:(c.get(r,0)+alpha)/(tot+alpha*N) for r in R}
        if kind=="nb":
            ch=hr.get(h,{}); th=sum(ch.values()); ct=tc.get(t,{}); tt=sum(ct.values())
            return {r:((ch.get(r,0)+alpha)/(th+alpha*N))*((ct.get(r,0)+alpha)/(tt+alpha*N)) for r in R}
    out={k:[] for k in ["freq","p_r_h","p_r_t","nb"]}
    gold_ok=[]
    for h,r,t in te:
        if r not in Rset:  # unseen-type gold excluded, matches neural filtered eval scope
            gold_ok.append(False); 
            for k in out: out[k].append(np.nan)
            continue
        gold_ok.append(True)
        known=pair_true.get((h,t), set())
        keep=set(R) - (known - {r})   # filtered: drop other known-true for this pair
        for k in out:
            out[k].append(expected_rank(scores_for(h,t,k), r, keep))
    return {k:np.array(v,dtype=float) for k,v in out.items()}, np.array(gold_ok)

In [ ]:
# === Paired bootstrap CIs for MRR differences (per query) ===
rng = np.random.default_rng(12345)
def paired_ci(rr_a, rr_b, n_boot=5000):
    # rr_*: per-query reciprocal ranks (same queries, aligned). Returns mean diff + 95% CI.
    d = rr_a - rr_b
    d = d[~np.isnan(d)]
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    boot = d[idx].mean(axis=1)
    return float(d.mean()), float(np.percentile(boot,2.5)), float(np.percentile(boot,97.5)), int((d>0).sum()), int((d<0).sum()), len(d)

summary_rows=[]
group_rows=[]
for ds, runs in records.items():
    ds_dir = find_splits(ds)
    if ds_dir is None: print("WARN: splits for", ds, "not attached; skipping baselines/grouping for it"); 
    # neural per-query reciprocal ranks, averaged across available seeds
    def rr_mean(run):
        arrs=[]
        for seed,f in sorted(records[ds].get(run,{}).items()):
            fr = np.load(f)["filtered_ranks"].astype(float)
            arrs.append(1.0/fr)
        return np.mean(np.vstack(arrs),axis=0) if arrs else None
    rr={run:rr_mean(run) for run in ["B0","B2","H_ONLY"] if run in runs}
    # ---- neural-vs-neural paired tests (clean: identical protocol) ----
    if "B2" in rr and "B0" in rr:
        m,lo,hi,wp,wn,nq = paired_ci(rr["B2"], rr["B0"])
        summary_rows.append([ds,"B2 - B0 (context effect)",m,lo,hi,wp,wn,nq, "CI excludes 0" if lo>0 or hi<0 else "CI includes 0"])
    if "B2" in rr and "H_ONLY" in rr:
        m,lo,hi,wp,wn,nq = paired_ci(rr["B2"], rr["H_ONLY"])
        summary_rows.append([ds,"B2 - H_ONLY (tail-context effect)",m,lo,hi,wp,wn,nq, "CI excludes 0" if lo>0 or hi<0 else "CI includes 0"])
    # ---- learned-vs-statistics (B0 vs best training-free baseline) ----
    if ds_dir is not None and "B0" in rr:
        base, gold_ok = baseline_perquery(ds_dir)
        # align: neural ranks were computed on the same test.txt order but may EXCLUDE unseen-type gold.
        # Our neural filtered_ranks length == number of scored test queries. Baselines use gold_ok mask.
        best_name = max(["p_r_t","nb"], key=lambda k: np.nanmean(1.0/base[k]))
        rr_base_all = 1.0/base[best_name]
        # restrict to gold_ok queries for both sides (neural eval also scores only seen-type gold)
        rr_b0 = rr["B0"]
        if len(rr_b0)==int(gold_ok.sum()):
            rr_base = rr_base_all[gold_ok]
            m,lo,hi,wp,wn,nq = paired_ci(rr_b0, rr_base)
            summary_rows.append([ds,f"B0 - {best_name} (learned vs statistics)",m,lo,hi,wp,wn,nq,"CI excludes 0" if lo>0 or hi<0 else "CI includes 0"])
        else:
            print(f"NOTE {ds}: neural scored {len(rr_b0)} queries vs {int(gold_ok.sum())} baseline gold_ok; "
                  "reporting baseline MRR without strict pairing.")
        # ---- endpoint-uncertainty grouping (Reviewer 3, comment 1) ----
        tr=load_triples(ds_dir/"train.txt"); te=load_triples(ds_dir/"test.txt")
        tc=collections.defaultdict(set)
        for h,r,t in tr: tc[t].add(r)
        cand_counts=[len(tc.get(t,set())) for (_,_,t) in te]  # candidate relations seen with this tail
        cand_counts=np.array(cand_counts,float)[gold_ok] if len(gold_ok)==len(te) else np.array(cand_counts,float)
        if "B2" in rr and len(rr["B2"])==len(cand_counts):
            # buckets by tail ambiguity
            qs=np.quantile(cand_counts,[0.33,0.66])
            bucket=np.digitize(cand_counts,qs)  # 0=low,1=mid,2=high ambiguity
            for b,lbl in [(0,"low tail-ambiguity"),(1,"mid"),(2,"high tail-ambiguity")]:
                msk=bucket==b
                if msk.sum()==0: continue
                gain = float((rr["B2"][msk]-rr["B0"][msk]).mean())
                group_rows.append([ds,lbl,int(msk.sum()),float(cand_counts[msk].mean()),
                                   float(rr["B0"][msk].mean()),float(rr["B2"][msk].mean()),gain])

S=pd.DataFrame(summary_rows, columns=["Dataset","Comparison","MeanRRdiff","CI_low","CI_high","wins_+","wins_-","n_queries","Verdict"])
G=pd.DataFrame(group_rows, columns=["Dataset","TailAmbiguityBucket","n","MeanCandRel/tail","MRR_B0","MRR_B2","ContextGain(B2-B0)"])
S.to_csv(OUT/"paired_significance.csv", index=False)
G.to_csv(OUT/"endpoint_uncertainty_grouping.csv", index=False)
print("=== Paired significance (per-query bootstrap, 95% CI) ==="); print(S.to_string(index=False))
print("\n=== Context gain by tail-ambiguity bucket (tests Reviewer 3 comment 1) ==="); print(G.to_string(index=False))
print("\nWrote paired_significance.csv and endpoint_uncertainty_grouping.csv")